# flowerly baseline: gpt-2 and lfm

Without data explore, this is workign version for now...
train and test separately on each dataset. or = original, cg = generated.

[data and methods](https://github.com/PurplexyFlower/Fake-review#datasets)

## 1. load data

load both csv files and remove empty or duplicate reviews.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "requirements.txt").is_file())
data_dir = root / "data" / "Flowerly"
files = {
    "GPT-2": "modern_reviews_gpt2.csv",
    "LFM": "modern_reviews_lfm.csv",
}

reviews = {}
for generator, filename in files.items():
    df = pd.read_csv(data_dir / filename).dropna(subset=["text_"]).copy()
    df["text_"] = df["text_"].str.strip()
    df = df[df["text_"] != ""].drop_duplicates(subset="text_")
    reviews[generator] = df
    print(generator, df["label"].value_counts().to_dict())

GPT-2 {'CG': 20190, 'OR': 20189}
LFM {'CG': 20212, 'OR': 20211}


## 2. split and train

use 80% for training and 20% for testing. keep the class ratio similar in both splits. train tf-idf and logistic regression on the training text.

In [2]:
rows = []

for generator, df in reviews.items():
    labels = df["label"].map({"OR": 0, "CG": 1})
    x_train, x_test, y_train, y_test = train_test_split(
        df["text_"], labels, test_size=0.20, stratify=labels, random_state=42
    )

    model = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True),
        LogisticRegression(max_iter=1000),
    )
    model.fit(x_train, y_train)

    predictions = model.predict(x_test)
    scores = model.predict_proba(x_test)[:, 1]
    rows.append({
        "generator": generator,
        "n_train": len(y_train),
        "n_test": len(y_test),
        "accuracy": accuracy_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "roc_auc": roc_auc_score(y_test, scores),
    })

## 3. results

compare accuracy, f1 and roc-auc. each model is tested on the same generator it trained on.

In [3]:
results = pd.DataFrame(rows)
display(results.round(3))
results.to_csv(root / "results" / "flowerly_baseline.csv", index=False)

,generator,n_train,n_test,accuracy,f1,roc_auc
0,GPT-2,32303,8076,0.934,0.933,0.982
1,LFM,32338,8085,0.991,0.991,1.000
